In [1]:
!pip install langchain langchain-core langchain-community

  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
   ---------------------------------------- 0.0/558.3 kB ? eta -:--:--
   ------------------ --------------------- 262.1/558.3 kB ? eta -:--:--
   ------------------ --------------------- 262.1/558.3 kB ? eta -:--:--
   ------------------ --------------------- 262.1/558.3 kB ? eta -:--:--
   -------------------------------------- 558.3/558.3 kB 423.5 kB/s eta 0:00:00
   ---------------------------------------- 0.0/654.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/654.0 kB ? eta -:--:--
   ---------------- ----------------------- 262.1/654.0 kB ? eta -:--:--
   ---------------- ----------------------- 262.1/654.0 kB ? eta -:--:--
   ---------------- ----------------------- 262.1/654.0 kB ? eta -:--:--
   ---------------- ----------------------- 262.1/654.0 kB ? eta -:--:--
   ------------------------------ ------- 524.3

In [2]:
!pip install pypdf pymupdf sentence-transformers chromadb

  Using cached pyyaml-6.0.3-cp313-cp313-win_amd64.whl.metadata (2.4 kB)
   ---------------------------------------- 0.0/19.8 MB ? eta -:--:--
   --- ------------------------------------ 1.8/19.8 MB 8.8 MB/s eta 0:00:03
   ------- -------------------------------- 3.7/19.8 MB 8.8 MB/s eta 0:00:02
   ----------- ---------------------------- 5.5/19.8 MB 8.6 MB/s eta 0:00:02
   -------------- ------------------------- 7.1/19.8 MB 8.5 MB/s eta 0:00:02
   ------------------ --------------------- 9.2/19.8 MB 8.6 MB/s eta 0:00:02
   --------------------- ------------------ 10.7/19.8 MB 8.4 MB/s eta 0:00:02
   ------------------------ --------------- 12.1/19.8 MB 8.3 MB/s eta 0:00:01
   ------------------------- -------------- 12.6/19.8 MB 7.7 MB/s eta 0:00:01
   ---------------------------- ----------- 14.2/19.8 MB 7.4 MB/s eta 0:00:01
   ------------------------------ --------- 15.2/19.8 MB 7.2 MB/s eta 0:00:01
   -------------------------------- ------- 16.3/19.8 MB 7.0 MB/s eta 0:00:01
   --

In [3]:
from langchain_core.documents import Document

sample_doc=Document(
    page_content="Hello Bhoomi",
    metadata={"source":"www.google.com"}
)

In [4]:
sample_doc

Document(metadata={'source': 'www.google.com'}, page_content='Hello Bhoomi')

In [5]:
type(sample_doc)

langchain_core.documents.base.Document

In [ ]:
#Text data-->document
# from langchain_community.document_loaders.text import TextLoader
# loader=TextLoader("./data/Python.txt",encoding="utf-8")
# document=loader.load()

In [ ]:
# document

[Document(metadata={'source': './data/Python.txt'}, page_content='Python is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, which makes it versatile for a wide range of applications.\nSome key features and benefits of Python include:\n* Ease of Learning: Simple syntax and readability make Python beginner-friendly.\n* Versatility: Suitable for web development, data analysis, artificial intelligence, machine learning, scientific computing, automation, and 

In [ ]:
#PDF data-->document (we can use PYPDFLoader for simple pdfs and PYMUPDFLoader for complex pdfs)
# from langchain_community.document_loaders.pdf import PyMuPDFLoader
# pdf_loader=PyMuPDFLoader("./data/research.pdf")
# pdf_doc=pdf_loader.load()
# pdf_doc

[Document(metadata={'producer': 'pdfcpu v0.12.1 dev', 'creator': '', 'creationdate': '2026-07-14T17:48:53+00:00', 'source': './data/research.pdf', 'file_path': './data/research.pdf', 'total_pages': 11, 'format': 'PDF 1.7', 'title': 'Attention is All you Need', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'keywords': '', 'moddate': '2026-07-14T17:48:53+00:00', 'trapped': '', 'modDate': "D:20260714174853+00'00'", 'creationDate': "D:20260714174853+00'00'", 'page': 0}, page_content='Attention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle

In [11]:
###Ingestion pipeline
#Data-->documents
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

In [14]:
def load_all_pdfs():
    folder_path="./data/pdfs"
    num_docs=0
    all_docs=[]

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"): #example research1.pdf
            #complete file path
            pdf_path=os.path.join(folder_path,filename) #it joins as "./data/pdfs/research1.pdf"

            loader=PyPDFLoader(pdf_path)
            doc=loader.load()

            num_docs+=1
            all_docs.extend(doc)

    print(f"total pdfs: ",num_docs)
    print(f"total_pages: ",len(all_docs))
    return all_docs

In [15]:
all_pdf_documents=load_all_pdfs()

total pdfs:  2
total_pages:  25


In [16]:
#chunks
!pip install langchain_text_splitters

In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents,chunk_size=500,chunk_overlap=50):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_overlap=chunk_overlap,
        chunk_size=chunk_size
    )

    chunked_docs=text_splitter.split_documents(documents)
    return chunked_docs

In [20]:
chunks=split_docs(all_pdf_documents)

In [21]:
len(chunks)

284

In [22]:
#Embedding
from sentence_transformers import SentenceTransformer

In [23]:
class EmbeddingManager:
    def __init__(self,model_name="all-MiniLM-L6-v2"):
        self.model_name=model_name
        print("Loading model...",self.model_name)

        self.model=SentenceTransformer(self.model_name)
        print("embedding dimensions: ",self.model.get_sentence_embedding_dimension())

    def generate_embedding(self,text):
        embeddings=self.model.encode(text,show_progress_bar=True)
        print("embedding shape: ",embeddings.shape)
        return embeddings

In [24]:
embedding_manager=EmbeddingManager()

Loading model... all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\RADHAGOPINATH\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\RADHAGOPINATH\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

embedding dimensions:  384


C:\Users\RADHAGOPINATH\AppData\Local\Temp\ipykernel_18680\3478893907.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions: ",self.model.get_sentence_embedding_dimension())
